In [ ]:
# Phase 1: 环境准备与数据探索分析

**目的**: 深入分析5G-NIDD数据集结构，验证数据完整性，建立预处理基线

**输入**: `data/raw/BTS1_BTS2_fields_preserved.csv`（1,215,890条记录）

**输出**: 数据质量报告和可视化图表


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('ggplot')
sns.set_palette("husl")

print("Phase 1: 数据探索分析环境准备完成")


Phase 1: 数据探索分析环境准备完成


In [ ]:
## 1.1 数据加载与基本信息


In [2]:
# 加载主数据集
data_path = '../data/raw/BTS1_BTS2_fields_preserved.csv'
print(f"正在加载数据: {data_path}")

df = pd.read_csv(data_path, low_memory=False)
print(f"数据加载完成!")
print(f"数据形状: {df.shape}")
print(f"内存使用: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


正在加载数据: ../data/raw/BTS1_BTS2_fields_preserved.csv
数据加载完成!
数据形状: (1215890, 113)
内存使用: 2282.35 MB


In [3]:
# 验证数据完整性
expected_records = 1215890
actual_records = len(df)

print(f"预期记录数: {expected_records:,}")
print(f"实际记录数: {actual_records:,}")
print(f"记录完整性: {'✓ 通过' if actual_records == expected_records else '✗ 不匹配'}")

# 显示字段信息
print(f"\n字段总数: {len(df.columns)}")
print("\n前10个字段:")
for i, col in enumerate(df.columns[:10]):
    print(f"{i+1:2d}. {col}")


预期记录数: 1,215,890
实际记录数: 1,215,890
记录完整性: ✓ 通过

字段总数: 113

前10个字段:
 1. Unnamed: 0
 2. SrcId
 3. Rank
 4. StartTime
 5. LastTime
 6. Trans
 7. Flgs
 8. Seq
 9. Dur
10. RunTime


In [ ]:
## 1.2 攻击类型分布分析


In [4]:
# 攻击类型统计
attack_type_counts = df['Attack Type'].value_counts()
print("攻击类型分布:")
print(attack_type_counts)

# 验证预期分布
expected_distribution = {
    'Benign': 477737,
    'UDPFlood': 457340,
    'HTTPFlood': 140812,
    'SlowrateDoS': 73124,
    'TCPConnectScan': 20052,
    'SYNScan': 20043,
    'UDPScan': 15906,
    'SYNFlood': 9721,
    'ICMPFlood': 1155
}

print("\n分布验证:")
for attack_type, expected_count in expected_distribution.items():
    actual_count = attack_type_counts.get(attack_type, 0)
    status = "✓" if actual_count == expected_count else "✗"
    print(f"{status} {attack_type}: {actual_count:,} (预期: {expected_count:,})")


攻击类型分布:
Attack Type
Benign            477737
UDPFlood          457340
HTTPFlood         140812
SlowrateDoS        73124
TCPConnectScan     20052
SYNScan            20043
UDPScan            15906
SYNFlood            9721
ICMPFlood           1155
Name: count, dtype: int64

分布验证:
✓ Benign: 477,737 (预期: 477,737)
✓ UDPFlood: 457,340 (预期: 457,340)
✓ HTTPFlood: 140,812 (预期: 140,812)
✓ SlowrateDoS: 73,124 (预期: 73,124)
✓ TCPConnectScan: 20,052 (预期: 20,052)
✓ SYNScan: 20,043 (预期: 20,043)
✓ UDPScan: 15,906 (预期: 15,906)
✓ SYNFlood: 9,721 (预期: 9,721)
✓ ICMPFlood: 1,155 (预期: 1,155)


In [ ]:
# 协议分布分析
proto_counts = df['Proto'].value_counts()
print("协议分布:")
print(proto_counts)
print(f"\n协议总数: {len(proto_counts)}")

# 攻击工具分布
tool_counts = df['Attack Tool'].value_counts()
print("\n攻击工具分布:")
print(tool_counts)

# IP地址模式分析
print("\n源IP地址样例 (前10个):")
print(df['SrcAddr'].head(10).tolist())
print("\n目标IP地址样例 (前10个):")
print(df['DstAddr'].head(10).tolist())


In [ ]:
# 数据质量检测
import os

def check_data_quality(df):
    """检测数据质量问题"""
    quality_report = {}
    
    # 缺失值检测
    missing_count = df.isnull().sum()
    quality_report['missing_values'] = missing_count[missing_count > 0].to_dict()
    
    # 重复记录检测
    duplicates = df.duplicated().sum()
    quality_report['duplicate_records'] = int(duplicates)
    
    # 异常值检测 (数值字段)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    outliers = {}
    for col in numeric_cols:
        if col != 'Unnamed: 0':  # 跳过索引列
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outlier_count = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
            if outlier_count > 0:
                outliers[col] = int(outlier_count)
    quality_report['outliers'] = outliers
    
    return quality_report

# 执行质量检测
quality_report = check_data_quality(df)
print("数据质量检测结果:")
print(f"缺失值字段数: {len(quality_report['missing_values'])}")
print(f"重复记录数: {quality_report['duplicate_records']}")
print(f"存在异常值的字段数: {len(quality_report['outliers'])}")

# 数据清洗
print("\n执行数据清洗...")
df_clean = df.copy()

# 删除重复记录
df_clean = df_clean.drop_duplicates()
print(f"删除重复记录后: {len(df_clean)} 条记录")

# 删除完全空的记录
df_clean = df_clean.dropna(how='all')
print(f"删除空记录后: {len(df_clean)} 条记录")


In [ ]:
# 生成可视化图表
import json
from datetime import datetime

plt.figure(figsize=(20, 15))

# 1. 攻击类型分布饼图
plt.subplot(2, 3, 1)
attack_counts = df_clean['Attack Type'].value_counts()
plt.pie(attack_counts.values, labels=attack_counts.index, autopct='%1.1f%%')
plt.title('攻击类型分布')

# 2. 协议分布柱状图
plt.subplot(2, 3, 2)
proto_counts = df_clean['Proto'].value_counts()
plt.bar(proto_counts.index, proto_counts.values)
plt.title('协议分布')
plt.xticks(rotation=45)

# 3. 攻击工具分布
plt.subplot(2, 3, 3)
tool_counts = df_clean['Attack Tool'].value_counts()
plt.bar(tool_counts.index, tool_counts.values)
plt.title('攻击工具分布')
plt.xticks(rotation=45)

# 4. 数据包数量分布（对数尺度）
plt.subplot(2, 3, 4)
plt.hist(np.log10(df_clean['TotPkts'] + 1), bins=50, alpha=0.7)
plt.title('数据包数量分布 (log10)')
plt.xlabel('log10(TotPkts + 1)')

# 5. 字节数分布（对数尺度）
plt.subplot(2, 3, 5)
plt.hist(np.log10(df_clean['TotBytes'] + 1), bins=50, alpha=0.7)
plt.title('字节数分布 (log10)')
plt.xlabel('log10(TotBytes + 1)')

# 6. 流持续时间分布
plt.subplot(2, 3, 6)
dur_values = pd.to_numeric(df_clean['Dur'], errors='coerce')
plt.hist(dur_values.dropna(), bins=50, alpha=0.7)
plt.title('流持续时间分布')
plt.xlabel('Duration (s)')

plt.tight_layout()
plt.savefig('../data/processed/metadata/data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 保存清洗后的数据
print("\\n保存清洗后的数据...")
df_clean.to_parquet('../data/processed/clean_data.parquet', index=False)
print(f"清洗后数据已保存: {len(df_clean)} 条记录")

# 生成数据探索报告
exploration_report = {
    "timestamp": datetime.now().isoformat(),
    "original_records": len(df),
    "cleaned_records": len(df_clean),
    "total_features": len(df_clean.columns),
    "attack_types": attack_counts.to_dict(),
    "protocol_distribution": proto_counts.to_dict(),
    "tool_distribution": tool_counts.to_dict(),
    "data_quality": quality_report,
    "memory_usage_mb": float(df_clean.memory_usage(deep=True).sum() / 1024**2)
}

# 保存报告
with open('../data/processed/metadata/data_exploration_report.json', 'w', encoding='utf-8') as f:
    json.dump(exploration_report, f, indent=2, ensure_ascii=False)

print("\\n✓ Phase 1 完成:")
print(f"  - 清洗后数据: ../data/processed/clean_data.parquet")
print(f"  - 探索报告: ../data/processed/metadata/data_exploration_report.json")
print(f"  - 可视化图表: ../data/processed/metadata/data_distribution.png")
